# 💳 Nettoyage complet des données - Paiements

Ce notebook contient le script de nettoyage de la table des paiements (`olist_order_payments_dataset.csv`).


## 1. Configuration des dossiers et du fichier

Initialisation de l'environnement d'exécution.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import os

RAW_DIR = Path(r"C:\Users\malek\OneDrive\Desktop\ESPRIT\3ème IA\PROJET BI\data\raw")
PROCESSED_DIR = RAW_DIR.parent / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FILE_NAME = "olist_order_payments_dataset.csv"
input_path = RAW_DIR / FILE_NAME
output_path = PROCESSED_DIR / f"{input_path.stem}_cleaned.csv"


## 2. Vérification et lecture du fichier

Inspection globale du fichier de paiement.


In [ ]:
if not input_path.exists():
    raise FileNotFoundError(f"Fichier introuvable : {input_path}")

df = pd.read_csv(input_path, encoding="utf-8-sig")

print("Dimensions initiales :", df.shape)
print("\nColonnes initiales :")
print(df.columns.tolist())
print("\nInformations générales :")
df.info()
print("\nAperçu des données :")
print(df.head())


## 3. Nettoyage des noms des colonnes

Sanitisation des noms de colonnes.


In [ ]:
df.columns = df.columns.str.replace(r"[^\w]", "", regex=True)

print("\n" + "=" * 70)
print("NETTOYAGE DES NOMS DES COLONNES")
print("=" * 70)
print(df.columns.tolist())


## 4. Nettoyage des colonnes textuelles

Sanitisation de `order_id` et `payment_type`.


In [ ]:
colonnes = ["order_id", "payment_type"]

for col in colonnes:
    df[col] = df[col].str.strip()
    df[col] = df[col].replace("", pd.NA)
    df[col] = df[col].str.replace(r"[^\w]", "", regex=True)


## 5. Suppression des colonnes inutiles

Suppression du champ `payment_sequential`.


In [ ]:
columns_to_remove = ["payment_sequential"]

df = df.drop(columns=columns_to_remove)

print("\nColonnes restantes :")
print(df.columns.tolist())


## 6. Suppression des lignes et colonnes vides

Vidage des entrées entièrement vides.


In [ ]:
df = df.dropna(axis=0, how="all")
df = df.dropna(axis=1, how="all")


## 7. Suppression des doublons exacts

Suppression des doublons.


In [ ]:
duplicates = df.duplicated().sum()
df = df.drop_duplicates()


## 8. Détection des valeurs aberrantes

Détection des valeurs aberrantes sur la colonne `payment_value` par méthode IQR.


In [ ]:
colonnes = ["payment_value"]

for col in colonnes:
    serie = pd.to_numeric(df[col], errors="coerce").dropna()
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    outliers = serie[(serie < limite_inf) | (serie > limite_sup)]

    print(f"\nColonne : {col}")
    print("Valeurs négatives :")
    print(df[df[col] < 0])
    print("Valeur minimale :", df[col].min())
    print("Valeur maximale :", df[col].max())
    print("Limite inférieure :", limite_inf)
    print("Limite supérieure :", limite_sup)
    print("Nombre d'outliers :", len(outliers))
    print("Pourcentage :", round(len(outliers) / len(serie) * 100, 2), "%")


## 9. Types de données

Affichage des types finaux.


In [ ]:
print("\nTypes de données :")
print(df.dtypes)


## 10. Bilan du nettoyage

Synthèse du nettoyage.


In [ ]:
print("\n" + "=" * 70)
print("BILAN DU NETTOYAGE")
print("=" * 70)
print("Nettoyage terminé.")
print("\nDoublons supprimés :", duplicates)
print("\nValeurs manquantes par colonne :")
print(df.isna().sum())
print("\nDimensions finales :", df.shape)


## 11. Exportation du fichier nettoyé

Sauvegarde sous `processed`.


In [ ]:
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nCopie enregistrée :", output_path)
print("Le fichier original reste intact.")
